# 04 — Scaling dhe Train/Validation/Test Split

Ky notebook kryen:
1. Group-aware split (60/20/20)
2. Filtrimin e price outliers pas split-it
3. Krijimin e target kategorik (`price_class`)
4. Feature engineering final (KMeans, log, ratio, location features) — mësohet vetëm nga train set
5. StandardScaler — fit vetëm mbi train
6. Ruajtjen e të gjitha CSV-ve finale

**Rregulli themelor:** asnjë transformim nuk mësohet nga validation/test set.

## 1. Importet dhe ngarkimi

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os

data_cleaned   = pd.read_csv("../../data/processed/data_featured.csv")
property_groups = pd.read_csv("../../data/processed/property_groups.csv").squeeze()

print("Shape:", data_cleaned.shape)

Shape: (21577, 25)


## 2. Ndarja X / y

In [2]:
feature_cols = [col for col in data_cleaned.columns if col != "price"]
X = data_cleaned[feature_cols]
y = data_cleaned["price"]

print("Features:", X.shape[1])
print("Samples: ", X.shape[0])
print("Kolonat:", list(X.columns))

Features: 24
Samples:  21577
Kolonat: ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'sale_year', 'sale_month', 'house_age_at_sale', 'was_renovated', 'years_since_renovation', 'has_basement']


## 3. Group-aware Train/Validation/Test Split (60/20/20)

In [3]:
groups = property_groups.loc[X.index]

test_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_val_idx, test_idx = next(test_splitter.split(X, y, groups=groups))

X_train_val = X.iloc[train_val_idx]
X_test      = X.iloc[test_idx]
y_train_val = y.iloc[train_val_idx]
y_test      = y.iloc[test_idx]
groups_train_val = groups.iloc[train_val_idx]

val_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(val_splitter.split(X_train_val, y_train_val, groups=groups_train_val))

X_train = X_train_val.iloc[train_idx]
X_val   = X_train_val.iloc[val_idx]
y_train = y_train_val.iloc[train_idx]
y_val   = y_train_val.iloc[val_idx]
groups_train = groups_train_val.iloc[train_idx]
groups_val   = groups_train_val.iloc[val_idx]
groups_test  = groups.iloc[test_idx]

print(f"Training set:   {X_train.shape[0]} rreshta ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} rreshta ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]} rreshta ({X_test.shape[0]/len(X)*100:.1f}%)")
print("\nOverlap mes seteve:")
print("  train/test:", len(set(groups_train) & set(groups_test)))
print("  train/val: ", len(set(groups_train) & set(groups_val)))
print("  val/test:  ", len(set(groups_val)   & set(groups_test)))

Training set:   12940 rreshta (60.0%)
Validation set: 4315 rreshta (20.0%)
Test set:       4322 rreshta (20.0%)

Overlap mes seteve:
  train/test: 0
  train/val:  0
  val/test:   0


## 4. Filtrimi i price outliers (pas split-it)

Kufiri p99 mësohet **vetëm nga y_train** dhe aplikohet mbi të gjitha setet.

In [4]:
train_price_99 = y_train.quantile(0.99)

train_mask = y_train <= train_price_99
val_mask   = y_val   <= train_price_99
test_mask  = y_test  <= train_price_99

before_train, before_val, before_test = len(y_train), len(y_val), len(y_test)

X_train = X_train.loc[train_mask]; y_train = y_train.loc[train_mask]
X_val   = X_val.loc[val_mask];     y_val   = y_val.loc[val_mask]
X_test  = X_test.loc[test_mask];   y_test  = y_test.loc[test_mask]

print(f"Kufiri p99 (nga train set): ${train_price_99:,.0f}")
print(f"Larguar nga train: {before_train - len(y_train)}")
print(f"Larguar nga val:   {before_val   - len(y_val)}")
print(f"Larguar nga test:  {before_test  - len(y_test)}")

Kufiri p99 (nga train set): $1,953,050
Larguar nga train: 130
Larguar nga val:   48
Larguar nga test:  43


## 5. Krijimi i target kategorik (`price_class`)

`qcut` mësohet vetëm mbi train set. Të njëjtët kufij aplikohen mbi val dhe test.

In [5]:
y_train_cat, bin_edges = pd.qcut(
    y_train, q=3, labels=["Low", "Medium", "High"], retbins=True
)
bin_edges[0]  = -float("inf")
bin_edges[-1] =  float("inf")

y_val_cat  = pd.cut(y_val,  bins=bin_edges, labels=["Low","Medium","High"], include_lowest=True)
y_test_cat = pd.cut(y_test, bins=bin_edges, labels=["Low","Medium","High"], include_lowest=True)

print(f"Low:    < ${bin_edges[1]:,.0f}")
print(f"Medium:   ${bin_edges[1]:,.0f} - ${bin_edges[2]:,.0f}")
print(f"High:   > ${bin_edges[2]:,.0f}")
print("\nShpërndarje train:")
print(y_train_cat.value_counts().sort_index())
print("\nShpërndarje val:")
print(y_val_cat.value_counts().sort_index())
print("\nShpërndarje test:")
print(y_test_cat.value_counts().sort_index())

Low:    < $360,000
Medium:   $360,000 - $560,000
High:   > $560,000

Shpërndarje train:
price
Low       4307
Medium    4279
High      4224
Name: count, dtype: int64

Shpërndarje val:
price
Low       1442
Medium    1390
High      1435
Name: count, dtype: int64

Shpërndarje test:
price
Low       1461
Medium    1412
High      1406
Name: count, dtype: int64


## 6. Feature engineering final (KMeans, log, ratio, location)

Të gjitha parametrat mësohen vetëm nga X_train.

In [6]:
SEATTLE_CENTER  = (47.6062, -122.3321)
BELLEVUE_CENTER = (47.6101, -122.2015)

def distance_to_point(lat, lng, clat, clng):
    lat_km = (lat - clat) * 111
    lng_km = (lng - clng) * 111 * np.cos(np.radians(clat))
    return np.sqrt(lat_km**2 + lng_km**2)

def add_features(df, lat_bins, long_bins, kmeans):
    df = df.copy()
    df["log_sqft_living"]   = np.log1p(df["sqft_living"])
    df["log_sqft_lot"]      = np.log1p(df["sqft_lot"])
    df["log_sqft_living15"] = np.log1p(df["sqft_living15"])
    df["log_sqft_lot15"]    = np.log1p(df["sqft_lot15"])
    df["bathrooms_per_bedroom"]   = df["bathrooms"] / df["bedrooms"].replace(0, np.nan)
    df["sqft_living_per_bedroom"] = df["sqft_living"] / df["bedrooms"].replace(0, np.nan)
    df["sqft_living_per_floor"]   = df["sqft_living"] / df["floors"].replace(0, np.nan)
    df["living_lot_ratio"]        = df["sqft_living"] / df["sqft_lot"].replace(0, np.nan)
    df["basement_ratio"]          = df["sqft_basement"] / df["sqft_living"].replace(0, np.nan)
    df["renovated_recently"]      = ((df["was_renovated"]==1) & (df["years_since_renovation"]<=10)).astype(int)
    df["grade_sqft_living"]       = df["grade"] * df["sqft_living"]
    df["view_waterfront"]         = df["view"] * df["waterfront"]
    df["lat_long"]                = df["lat"] * df["long"]
    df["distance_to_seattle"]     = distance_to_point(df["lat"], df["long"], *SEATTLE_CENTER)
    df["distance_to_bellevue"]    = distance_to_point(df["lat"], df["long"], *BELLEVUE_CENTER)
    df["min_distance_to_center"]  = df[["distance_to_seattle","distance_to_bellevue"]].min(axis=1)
    df["lat_bin"]          = pd.cut(df["lat"],  bins=lat_bins,  labels=False, include_lowest=True)
    df["long_bin"]         = pd.cut(df["long"], bins=long_bins, labels=False, include_lowest=True)
    df["location_cluster"] = kmeans.predict(df[["lat","long"]])
    df["is_luxury_grade"]  = (df["grade"] >= 10).astype(int)
    df["is_high_view"]     = (df["view"]  >= 3).astype(int)
    df["is_premium"]       = ((df["grade"]>=10)|(df["waterfront"]==1)|(df["view"]>=3)).astype(int)
    df["waterfront_grade"]       = df["waterfront"] * df["grade"]
    df["waterfront_sqft_living"] = df["waterfront"] * df["sqft_living"]
    df["view_grade"]             = df["view"] * df["grade"]
    df["view_sqft_living"]       = df["view"] * df["sqft_living"]
    df["grade_lat_bin"]          = df["grade"] * df["lat_bin"]
    df["grade_location_cluster"] = df["grade"] * df["location_cluster"]
    return df.fillna(0)

# Mëso parametrat vetëm nga X_train
lat_bins  = np.linspace(X_train["lat"].min(),  X_train["lat"].max(),  11)
long_bins = np.linspace(X_train["long"].min(), X_train["long"].max(), 11)

kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
kmeans.fit(X_train[["lat","long"]])

X_train = add_features(X_train, lat_bins, long_bins, kmeans)
X_val   = add_features(X_val,   lat_bins, long_bins, kmeans)
X_test  = add_features(X_test,  lat_bins, long_bins, kmeans)

feature_cols = list(X_train.columns)
print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)

X_train shape: (12810, 52)
X_val shape:   (4267, 52)
X_test shape:  (4279, 52)


## 7. StandardScaler

`fit()` vetëm mbi X_train. `transform()` mbi val dhe test.

In [7]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val),       columns=feature_cols, index=X_val.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols, index=X_test.index)

print("Pas StandardScaler (mean dhe std per 5 features):")
print(X_train_scaled.describe().round(3).loc[["mean","std"]].T.head(5))
print("\n✓ Mean ≈ 0 dhe Std ≈ 1 per te gjitha features")

Pas StandardScaler (mean dhe std per 5 features):
             mean  std
bedrooms      0.0  1.0
bathrooms    -0.0  1.0
sqft_living   0.0  1.0
sqft_lot     -0.0  1.0
floors        0.0  1.0

✓ Mean ≈ 0 dhe Std ≈ 1 per te gjitha features


## 8. Ruajtja e të gjitha CSV-ve finale

In [8]:
os.makedirs("../../data/processed", exist_ok=True)

def build_clf(X, y):
    d = X.reset_index(drop=True).copy()
    d["price_class"] = y.reset_index(drop=True)
    return d

def build_reg(X, y):
    d = X.reset_index(drop=True).copy()
    d["price"] = y.reset_index(drop=True)
    return d

# Classification datasets
build_clf(X_train_scaled, y_train_cat).to_csv("../../data/processed/train_dataset.csv",          index=False)
build_clf(X_val_scaled,   y_val_cat).to_csv(  "../../data/processed/val_dataset.csv",            index=False)
build_clf(X_test_scaled,  y_test_cat).to_csv( "../../data/processed/test_dataset.csv",           index=False)
build_clf(X_train, y_train_cat).to_csv(        "../../data/processed/train_unscaled_dataset.csv", index=False)
build_clf(X_val,   y_val_cat).to_csv(          "../../data/processed/val_unscaled_dataset.csv",   index=False)
build_clf(X_test,  y_test_cat).to_csv(         "../../data/processed/test_unscaled_dataset.csv",  index=False)

# Regression datasets
build_reg(X_train_scaled, y_train).to_csv("../../data/processed/train_regression_dataset.csv",          index=False)
build_reg(X_val_scaled,   y_val).to_csv(  "../../data/processed/val_regression_dataset.csv",            index=False)
build_reg(X_test_scaled,  y_test).to_csv( "../../data/processed/test_regression_dataset.csv",           index=False)
build_reg(X_train, y_train).to_csv(        "../../data/processed/train_regression_unscaled_dataset.csv", index=False)
build_reg(X_val,   y_val).to_csv(          "../../data/processed/val_regression_unscaled_dataset.csv",   index=False)
build_reg(X_test,  y_test).to_csv(         "../../data/processed/test_regression_unscaled_dataset.csv",  index=False)

# Bin edges per reference
np.save("../../data/processed/price_bin_edges.npy", bin_edges)

print("✓ Skedarët e ruajtur në data/processed/:")
files = [
    "train_dataset.csv","val_dataset.csv","test_dataset.csv",
    "train_unscaled_dataset.csv","val_unscaled_dataset.csv","test_unscaled_dataset.csv",
    "train_regression_dataset.csv","val_regression_dataset.csv","test_regression_dataset.csv",
    "train_regression_unscaled_dataset.csv","val_regression_unscaled_dataset.csv","test_regression_unscaled_dataset.csv",
    "price_bin_edges.npy"
]
for f in files:
    print(f"  {f}")

print(f"\nRezume finale:")
print(f"  Training samples:   {X_train_scaled.shape[0]}")
print(f"  Validation samples: {X_val_scaled.shape[0]}")
print(f"  Test samples:       {X_test_scaled.shape[0]}")
print(f"  Features:           {X_train_scaled.shape[1]}")
print(f"  Target classes:     Low / Medium / High")

✓ Skedarët e ruajtur në data/processed/:
  train_dataset.csv
  val_dataset.csv
  test_dataset.csv
  train_unscaled_dataset.csv
  val_unscaled_dataset.csv
  test_unscaled_dataset.csv
  train_regression_dataset.csv
  val_regression_dataset.csv
  test_regression_dataset.csv
  train_regression_unscaled_dataset.csv
  val_regression_unscaled_dataset.csv
  test_regression_unscaled_dataset.csv
  price_bin_edges.npy

Rezume finale:
  Training samples:   12810
  Validation samples: 4267
  Test samples:       4279
  Features:           52
  Target classes:     Low / Medium / High
